# Batch Supplier Assignment by SKU

Given a finished-good SKU, this notebook uses `db.batch()` (greedy set-cover) to assign every
BOM raw-material component to the **fewest possible suppliers** from the enriched supplier catalog.

**Workflow:**
1. **Browse BOMs** — list all finished-good SKUs and their components.
2. **Compare assignments** — `filter_products.compare_batch()` shows old BOM supplier vs new batch assignment side-by-side, with quality score and purity for each new assignment.
3. **Filtered assignment** — run the same comparison with quality/price constraints to restrict eligible suppliers.
4. **Before / after evaluation** — LLM judge (Gemini via DeepEval GEval) scores assignment quality with and without filters so you can quantify the improvement.

**Key functions (all importable from `filter_products`):**
| Function | Purpose |
|---|---|
| `make_filters(...)` | Build a composable filter list from range constraints |
| `rank_suppliers(sku)` | Batch + score, returns ranked assignments |
| `compare_batch(sku)` | Batch vs original BOM supplier, row per component |
| `print_batch_comparison(rows)` | Pretty-print compare_batch output |


## 1. Imports

In [4]:
import importlib
import sys
sys.path.insert(0, ".")

import db, filter_products
importlib.reload(db)
importlib.reload(filter_products)

from db import get_boms, get_bom_components
from filter_products import (
    make_filters,
    rank_suppliers,
    compare_batch,
    print_batch_comparison,
)


## 2. Browse BOMs

List all available finished-good SKUs. Each BOM maps a produced SKU to a set of raw-material
component SKUs (the ingredients). Change `PRODUCT_SKU` in the next section to any SKU shown here.


In [2]:
boms = get_boms()
print(f"{len(boms)} BOMs available")
boms[:5]


149 BOMs available


[{'BOMId': 48,
  'ProducedSKU': 'FG-amazon-b0002wrqy4',
  'CompanyName': 'ALL ONE'},
 {'BOMId': 57,
  'ProducedSKU': 'FG-amazon-b007vwbm2u',
  'CompanyName': 'Vitacost'},
 {'BOMId': 49,
  'ProducedSKU': 'FG-amazon-b012t9ga6c',
  'CompanyName': 'Optimum Nutrition'},
 {'BOMId': 61,
  'ProducedSKU': 'FG-amazon-b016mrfr8y',
  'CompanyName': 'Garden of Life'},
 {'BOMId': 59,
  'ProducedSKU': 'FG-amazon-b07dlflsyp',
  'CompanyName': "Simply Tera's"}]

## 3. Unfiltered batch assignment

`compare_batch()` runs greedy set-cover on the BOM and returns one row per component comparing:
- **old_supplier** — the company originally listed in the BOM data
- **new_supplier** — the supplier chosen by `batch()` from the enriched catalog
- **new_score** — composite quality score (0–1) for the new assignment; `None` if no enrichment data
- **new_purity** — purity fraction for the new assignment

Rows marked `*` are components where the supplier changed.

> `PRODUCT_SKU` defaults to the first BOM. Change it to any SKU from the list above.


In [3]:
PRODUCT_SKU = boms[0]["ProducedSKU"]   # ← change to any finished-good SKU

rows = compare_batch(PRODUCT_SKU)
if not rows:
    print(f"No BOM found for {PRODUCT_SKU!r}")
else:
    print_batch_comparison(rows, PRODUCT_SKU)


Product : FG-amazon-b0002wrqy4
Components: 21  |  Changed: 21  |  Uncovered: 0

SKU                                              Old Supplier             New Supplier               Score  Purity
------------------------------------------------------------------------------------------------------------------
RM-C2-ascorbic-acid-4823fabf                     ALL ONE                  Prinova USA                0.000    N/A  *
RM-C2-beta-carotene-bb90cd60                     ALL ONE                  AIDP                       0.000    N/A  *
RM-C2-choline-bitartrate-296d4da4                ALL ONE                  Prinova USA                0.000    N/A  *
RM-C2-cyanocobalamin-c2d9669b                    ALL ONE                  Prinova USA                0.000    N/A  *
RM-C2-d-alpha-tocopheryl-succinate-532e67fd      ALL ONE                  Prinova USA                0.000    N/A  *
RM-C2-d-calcium-pantothenate-bc49e976            ALL ONE                  Prinova USA                0.00

## 4. Filtered batch assignment

Pass `filters` to restrict eligible suppliers before the set-cover step.
Any supplier-product that fails a filter is excluded from consideration entirely —
the algorithm then picks the best remaining coverage.

Uncomment constraints as needed. Products missing a field pass through by default (opt-in).


In [4]:
filters = make_filters(
    # price_range    = (None, 200.0),   # max price per unit
    # quantity_range = (10.0, None),    # min order quantity
    # purity_range   = (0.95, None),    # min purity
    # quality_range  = (0.7,  None),    # min quality score (0.7=food, 0.9=GMP, 1.0=USP)
    # quality_metrics = {
    #     "heavy_metals":        (None, 10.0),
    #     "identity_confidence": (0.95, None),
    # },
)

rows_filtered = compare_batch(PRODUCT_SKU, filters=filters)
if not rows_filtered:
    print(f"No BOM found for {PRODUCT_SKU!r}")
else:
    print_batch_comparison(rows_filtered, f"{PRODUCT_SKU}  [filtered]")


Product : FG-amazon-b0002wrqy4  [filtered]
Components: 21  |  Changed: 21  |  Uncovered: 0

SKU                                              Old Supplier             New Supplier               Score  Purity
------------------------------------------------------------------------------------------------------------------
RM-C2-ascorbic-acid-4823fabf                     ALL ONE                  Prinova USA                0.000    N/A  *
RM-C2-beta-carotene-bb90cd60                     ALL ONE                  AIDP                       0.000    N/A  *
RM-C2-choline-bitartrate-296d4da4                ALL ONE                  Prinova USA                0.000    N/A  *
RM-C2-cyanocobalamin-c2d9669b                    ALL ONE                  Prinova USA                0.000    N/A  *
RM-C2-d-alpha-tocopheryl-succinate-532e67fd      ALL ONE                  Prinova USA                0.000    N/A  *
RM-C2-d-calcium-pantothenate-bc49e976            ALL ONE                  Prinova USA        

## 5. Before / after evaluation (LLM as judge)

`evaluate_batch()` samples BOMs at random, runs batch assignment on each, and scores every
supplier–component assignment using **Gemini as an LLM judge** (DeepEval GEval).

The judge reads the supplier's scraped HTML page and scores (0–1) whether that page confirms
the supplier actually sells the right raw material at appropriate supplement-grade quality.

Two passes are compared:
- **Before** — raw batch with no filters (maximize coverage, ignore quality constraints)
- **After** — batch with quality filters applied (may reduce coverage but improves match quality)

> Requires `GEMINI_API_KEY` in the environment. Assignments with no scraped HTML are skipped.
> Only ~200 of 1 600 supplier–product pairs have HTML pages (6 suppliers), so expect many skips.


In [7]:
import generate_synthetic_product_qualities
importlib.reload(generate_synthetic_product_qualities)

n = generate_synthetic_product_qualities.generate(seed=42)
print(f"Seeded synthetic prices for {n} supplier-product pairs.")

Seeded synthetic prices for 0 supplier-product pairs.


In [8]:
import evaluate
importlib.reload(evaluate)
from evaluate import evaluate_batch

N_SAMPLES = 5   # number of BOMs to sample per pass
SEED = 42       # fix seed for a fair before/after comparison

print("=" * 60)
print("BEFORE — unfiltered batch")
print("=" * 60)
results_before = evaluate_batch(n_samples=N_SAMPLES, filters=None, seed=SEED)

print()
print("=" * 60)
print("AFTER  — filtered batch")
print("=" * 60)
eval_filters = make_filters(
    quality_range=(0.7, None),
    # purity_range=(0.95, None),
)
results_after = evaluate_batch(n_samples=N_SAMPLES, filters=eval_filters, seed=SEED)

# ── Summary ──────────────────────────────────────────────────────────────────
def _mean(results):
    scored = [r["mean_score"] for r in results if r["mean_score"] is not None]
    return sum(scored) / len(scored) if scored else None

before_mean = _mean(results_before)
after_mean  = _mean(results_after)

print()
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  Before (unfiltered): {before_mean:.3f}" if before_mean is not None else "  Before: n/a")
print(f"  After  (filtered)  : {after_mean:.3f}"  if after_mean  is not None else "  After : n/a")
if before_mean is not None and after_mean is not None:
    delta = after_mean - before_mean
    print(f"  Delta              : {delta:+.3f}")


BEFORE — unfiltered batch

--- FG-cvs-448437 ---


  RM-C37-gelatin-a40fdc3f → Capsuline: 1.00 (pass) — The input specifies 'gelatin' as the raw material component. The retrieval context from Capsuline describes 'Clear Empty Gelatin Capsules Size 00' which are made from 100% bovine hide gelatin. These capsules are clearly presented as suitable for supplement manufacturing, mentioning 'high-speed filling machines' and 'wholesale pricing', indicating B2B supply. This is a very strong match for the requested component and its intended use.
  mean: 1.00  (31 skipped, no HTML)

--- FG-amazon-b07z2x2xtc ---
  mean: n/a  (3 skipped, no HTML)

--- FG-target-a-91641237 ---
  mean: n/a  (13 skipped, no HTML)

--- FG-target-a-78807251 ---
  mean: n/a  (12 skipped, no HTML)

--- FG-target-a-13943499 ---


  RM-C14-gelatin-017eb28d → Capsuline: 0.90 (pass) — The input specifies 'gelatin' as the raw material component SKU. The retrieval context from the supplier 'Capsuline' describes 'Clear Gelatin Capsules Size 0'. While not raw gelatin powder, these capsules are explicitly stated to be made from '100% bovine hide' (gelatin) and are designed for 'high-speed filling machines' and 'disintegrate within 15 minutes', indicating suitability for supplement manufacturing. This is a very close and suitable equivalent to sourcing gelatin for a supplement product.
  mean: 0.90  (14 skipped, no HTML)

Overall: 0.95/1.0 across 2 BOMs

AFTER  — filtered batch

--- FG-cvs-448437 ---


  RM-C37-gelatin-a40fdc3f → Capsuline: 0.20 (fail) — The input specifies the raw material component SKU `RM-C37-gelatin-a40fdc3f` (gelatin). However, the retrieval context from the supplier 'Capsuline' describes 'Clear Empty Gelatin Capsules'. While capsules are made from gelatin and used in supplement manufacturing, they are a finished product, not the raw material component 'gelatin' itself. Therefore, the context does not confirm the sale of the specified raw material in its raw form.
  mean: 0.20  (31 skipped, no HTML)

--- FG-amazon-b07z2x2xtc ---
  mean: n/a  (3 skipped, no HTML)

--- FG-target-a-91641237 ---
  mean: n/a  (13 skipped, no HTML)

--- FG-target-a-78807251 ---
  mean: n/a  (12 skipped, no HTML)

--- FG-target-a-13943499 ---


  RM-C14-gelatin-017eb28d → Capsuline: 0.10 (fail) — The input specifies the raw material component SKU `RM-C14-gelatin-017eb28d` for 'gelatin'. The actual output provides prices for 'gelatin' in kilograms. However, the retrieval context for the supplier 'Capsuline' describes 'Clear Gelatin Capsules Size 0', which are finished products made *from* gelatin, rather than raw bulk gelatin powder as a component. The context confirms the sale of gelatin capsules, not the raw material gelatin in a form suitable for manufacturing other supplements, leading to a mismatch between the desired component and the product offered in the context.
  mean: 0.10  (14 skipped, no HTML)

Overall: 0.15/1.0 across 2 BOMs

SUMMARY
  Before (unfiltered): 0.950
  After  (filtered)  : 0.150
  Delta              : -0.800
